# 01 — Data Setup, PORTEX Computation & Labeling

Loads the merged advisory–AIS dataset, recomputes the PORTEX index, creates binary event labels, and explores class imbalance.

> **Data**: Update `DATA_PATH` to point to your local copy of `master_port_storm_dataset_portex_encoded.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from src.data.preprocessing import compute_portex, create_event_labels, impute_medians, FEATURE_COLS

# ── Config ──────────────────────────────────────────────
DATA_PATH = "../data/raw/master_port_storm_dataset_portex_encoded.csv"
# DATA_PATH = "/content/drive/MyDrive/ais_data/master_port_storm_dataset_portex_encoded.csv"  # Colab

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()


In [ ]:
# ── Impute missing values ───────────────────────────────
df = impute_medians(df)
print("NaN count after imputation:", df.isna().sum().sum())


In [ ]:
# ── Compute PORTEX index ────────────────────────────────
df = compute_portex(df)
df[["PORT", "DATE", "PORTEX_risk_recalc"]].head()


In [ ]:
# ── Create event labels ─────────────────────────────────
df = create_event_labels(df, threshold_percentile=0.95)

print("Class balance:")
print(df["Event_Label"].value_counts())


In [ ]:
# ── Visualise PORTEX distribution ───────────────────────
threshold = df["PORTEX_risk_recalc"].quantile(0.95)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df["PORTEX_risk_recalc"], bins=40, kde=False, color="#2196F3", alpha=0.8, ax=ax)
ax.axvline(threshold, color="crimson", linestyle="--", label=f"95th pct = {threshold:.2f}")
ax.set(xlabel="PORTEX Risk (0–100)", ylabel="Frequency", title="Distribution of PORTEX Risk Scores")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("../results/figures/01_portex_distribution.png", dpi=150); plt.show()


In [ ]:
# ── Labeling sensitivity analysis ───────────────────────
rows = []
for pct in [0.90, 0.95, 0.975]:
    cut = df["PORTEX_risk_recalc"].quantile(pct)
    n = (df["PORTEX_risk_recalc"] >= cut).sum()
    rows.append({"Threshold Percentile": f"p{int(pct*100)}", "Cutoff": round(cut,3),
                 "Positives": n, "Positive Rate (%)": round(100*n/len(df), 2)})
pd.DataFrame(rows)


In [ ]:
# ── Events by port ──────────────────────────────────────
summary = df.groupby("PORT")["Event_Label"].agg(Events="sum", Total="count")
summary["Event Rate (%)"] = (100 * summary["Events"] / summary["Total"]).round(2)
print(summary)
